# 05 — SFLLD Missing-Value Treatment Verification

**Purpose:** independently verify that  
`SFLLD_modeling_dataset_missing_values_treated.parquet`  
was produced according to the frozen leakage-safe missing-value policy.

This notebook **does not modify or re-impute the dataset**. It only audits the saved result.

It verifies:

- the treated dataset is found in `data/output/final_modeling_dataset`;
- split structure and core identifiers are valid;
- all model fields that were supposed to be treated contain zero missing values;
- `Original_DTI` has a valid pre-imputation missing indicator;
- categorical missingness is represented explicitly as `"Missing"`;
- Strict/LS final HPI and BLS fields are complete after treatment;
- Regular-PIT final HPI/BLS fields are checked when present;
- `Property_Valuation_Method` is allowed to remain missing by design;
- raw/intermediate macro and audit fields are **not incorrectly interpreted as treatment failures**;
- imputation medians and indicator counts are compared against the treatment manifest when available;
- split-level missingness is reported separately for Train, Validation, OOT Test, and other periods;
- a final PASS/FAIL summary is generated.

A full DataFrame-wide `0 missing` requirement would be incorrect because several lineage/audit/excluded fields are intentionally retained with missing values.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 240)

FILE_NAME = "SFLLD_modeling_dataset_missing_values_treated.parquet"

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        candidate = p / "data" / "output" / "final_modeling_dataset" / FILE_NAME
        if candidate.exists():
            return p
    raise FileNotFoundError(
        f"Could not locate {FILE_NAME}. Run from the SFLLD project root or notebooks/."
    )

ROOT = find_project_root()
INPUT = ROOT / "data" / "output" / "final_modeling_dataset" / FILE_NAME
QA_DIR = ROOT / "data" / "output" / "quality_assurance" / "05_missing_value_treatment_verification"
QA_DIR.mkdir(parents=True, exist_ok=True)

TREATMENT_MANIFEST = (
    ROOT / "data" / "output" / "quality_assurance"
    / "04_missing_value_treatment" / "04_missing_value_treatment_manifest.json"
)

print("Project root       :", ROOT.resolve())
print("Treated dataset    :", INPUT.resolve())
print("QA output directory:", QA_DIR.resolve())
print("Treatment manifest :", TREATMENT_MANIFEST.resolve())
print("Manifest exists    :", TREATMENT_MANIFEST.exists())


Project root       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
Treated dataset    : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_missing_values_treated.parquet
QA output directory: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\05_missing_value_treatment_verification
Treatment manifest : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\04_missing_value_treatment\04_missing_value_treatment_manifest.json
Manifest exists    : True


## 1. Load treated dataset

In [2]:
df = pd.read_parquet(INPUT)

print("Shape:", df.shape)
print("\nSplit counts:")
display(df["Split"].value_counts(dropna=False).rename_axis("Split").reset_index(name="rows"))

print("\nFirst 5 rows:")
display(df.head())

required_core = {"Loan_ID", "Split", "Target_24M"}
missing_core = required_core - set(df.columns)
assert not missing_core, f"Missing required core columns: {sorted(missing_core)}"
assert df["Loan_ID"].notna().all(), "FAIL: Loan_ID contains missing values."
assert df["Loan_ID"].is_unique, "FAIL: Loan_ID is not unique."
assert df["Split"].notna().all(), "FAIL: Split contains missing values."
assert df["Target_24M"].notna().all(), "FAIL: Target_24M contains missing values."

split_norm = df["Split"].astype("string").str.strip().str.casefold()
train_mask = split_norm.eq("train")
assert train_mask.any(), "FAIL: no Train split found."

print("PASS — core dataset integrity checks.")


Shape: (713000, 110)

Split counts:


,Split,rows
0,Other Period,404240
1,Train,160003
2,OOT Test,99719
3,Validation,49038



First 5 rows:


,Loan_ID,Dataset_Vintage,FICO_Score,First_Payment_Date,First_Time_Homebuyer,Maturity_Date,MSA,MI_Percentage,Number_of_Units,Occupancy_Status,Original_CLTV,Original_DTI,Original_UPB,Original_LTV,Original_Interest_Rate,Channel,Prepayment_Penalty_Indicator,Amortization_Type,Property_State,Property_Type,Postal_Code,Loan_Purpose,Original_Loan_Term,Number_of_Borrowers,Seller_Name,Super_Conforming_Flag,Special_Eligibility_Program,HARP_Indicator,Property_Valuation_Method,Interest_Only_Indicator,Target_24M,Performance_Window_End,COVID_Window_Overlap,Split,PIT_FHFA_Quarter_Date,PIT_BLS_Month_Date,Freddie_State,Freddie_MSA,Freddie_ZIP3,FHFA_State_HPI,FHFA_State_HPI_Growth_12M,FHFA_ZIP3_HPI,FHFA_ZIP3_HPI_Growth_12M,FHFA_MSA_HPI,FHFA_MSA_HPI_Growth_12M,Housing_HPI,Housing_HPI_Growth_12M,Housing_Geography_Source,Housing_Fallback_Level,BLS_Unemployment_Rate,BLS_Employment,BLS_Labor_Force,BLS_Employment_Growth_12M,BLS_Unemployment_Rate_Change_12M,Labor_Geography_Source,Strict_PIT_FHFA_Quarter_Date,Strict_PIT_BLS_Month_Date,LS_Freddie_State,LS_Freddie_ZIP3,LS_Freddie_MSA,LS_FHFA_State_HPI,LS_FHFA_State_HPI_Growth_12M,LS_FHFA_ZIP3_HPI,LS_FHFA_ZIP3_HPI_Growth_12M,LS_FHFA_MSA_HPI,LS_FHFA_MSA_HPI_Growth_12M,LS_Housing_HPI,LS_Housing_HPI_Growth_12M,LS_Housing_Geography_Source,LS_Housing_Fallback_Level,LS_BLS_Unemployment_Rate,LS_BLS_Employment,LS_BLS_Labor_Force,LS_BLS_Employment_Growth_12M,LS_BLS_Unemployment_Rate_Change_12M,LS_Labor_Geography_Source,_State_FIPS,Strict_PIT_BLS_State_Month_Date,BLS_State_Assumed_Available_Date,_State_BLS_Unemployment_Rate,_State_BLS_Civilian_Labor_Force,_State_BLS_Employment,_State_BLS_Unemployment,_State_BLS_Labor_Force_Participation_Rate,_State_BLS_Employment_Population_Ratio,LS_BLS_Unemployment_Rate_State_Fallback,LS_BLS_Unemployment_Rate_Final,LS_BLS_Geography_Source_Final,LS_BLS_Fallback_Level,LS_BLS_State_Fallback_Applied,09G_BLS_Remaining_Missing_Reason,BLS_Unemployment_Rate_State_Fallback,BLS_Employment_Growth_12M_State_Fallback,BLS_Unemployment_Rate_Change_12M_State_Fallback,BLS_Unemployment_Rate_Final,BLS_Employment_Growth_12M_Final,BLS_Unemployment_Rate_Change_12M_Final,BLS_State_Fallback_Applied,BLS_Fallback_Level,BLS_Geography_Source_Final,BLS_Remaining_Missing_Reason,LS_BLS_Employment_Growth_12M_State_Fallback,LS_BLS_Unemployment_Rate_Change_12M_State_Fallback,LS_BLS_Employment_Growth_12M_Final,LS_BLS_Unemployment_Rate_Change_12M_Final,DTI_Missing,LS_Housing_ResidualMissing,LS_BLS_ResidualMissing,Housing_Regular_ResidualMissing,BLS_Regular_ResidualMissing
0,F10Q10000014,2010.0,784.0,2010-05-01,N,2025-04-01,45780,0.0,1.0,P,90.0,38.0,216000,80.0,4.375,R,N,FRM,OH,SF,436,N,180,2,OTHER,N,Missing,N,<NA>,N,0,2012-04-01,False,Other Period,2010-01-01,2010-04-01,OH,45780,436,243.57,-0.054428,133.19,-0.078142,135.30,-0.063344,135.30,-0.063344,MSA/MSAD,0,11.5,273584.0,309031.0,0.006267,-0.7,MSA/MSAD_EXACT,2009-10-01,2010-02-01,OH,436,45780,246.30,-0.026251,136.68,-0.017609,136.72,-0.033029,136.72,-0.033029,MSA/MSAD,0,12.0,272332.0,309542.0,-0.012564,0.3,MSA/MSAD_EXACT,39,2010-02-01,2010-04-14,10.7,5874284,5246299,627985,65.6,58.6,10.7,12.0,MSA_STRICT_EXISTING,MSA,0,NOT_MISSING,10.3,-0.012373,0.1,11.5,0.006267,-0.7,0,0,MSA/MSAD_REGULAR,NOT_MISSING,-0.027541,1.3,-0.012564,0.3,0,0,0,0,0
1,F10Q10000069,2010.0,795.0,2010-03-01,N,2025-02-01,Missing,0.0,1.0,P,67.0,35.0,200000,67.0,4.500,R,N,FRM,KS,SF,670,N,180,2,OTHER,N,Missing,N,<NA>,N,0,2012-02-01,False,Other Period,2009-10-01,2010-02-01,KS,<NA>,670,235.14,-0.004741,155.35,0.014630,NaN,NaN,155.35,0.014630,ZIP3,1,NaN,NaN,NaN,NaN,NaN,<NA>,2009-07-01,2010-02-01,KS,670,<NA>,235.59,-0.001060,154.32,-0.000453,NaN,NaN,154.32,-0.000453,ZIP3,1,NaN,NaN,NaN,NaN,NaN,<NA>,20,2009-12-01,2010-02-14,7.3,1507608,1397280,110328,70.4,65.2,7.3,7.3,STATE_STRICT_FALLBACK,STATE,1,NOT_MISSING,7.2,-0.015565,0.9,7.2,-0.015565,0.9,1,1,STATE_REGULAR,NOT_MISSING,-0.017360,1.7,-0.017360,1.7,0,0,0,0,0
2,F10Q10000089,2010.0,784.0,2010-03-01,N,2025-02-01,36100,0.0,1.0,P,55.0,47.0,146000,55.0,4.500,R,

PASS — core dataset integrity checks.


## 2. Frozen treatment groups

These are the fields that **must be complete after treatment** under our agreed policy.


In [3]:
LOAN_TRAIN_MEDIAN = [
    "FICO_Score",
    "MI_Percentage",
    "Number_of_Units",
    "Original_CLTV",
    "Original_LTV",
]

DTI_COL = "Original_DTI"
DTI_INDICATOR = "DTI_Missing"

CATEGORICAL_EXPLICIT_MISSING = [
    "First_Time_Homebuyer",
    "MSA",
]

HIGH_MISSING_ABLATION = [
    "Special_Eligibility_Program",
]

TEMPORAL_SUPPORT_EXCLUDE = [
    "Property_Valuation_Method",
]

STRICT_FINAL_HOUSING = [
    "LS_Housing_HPI",
    "LS_Housing_HPI_Growth_12M",
]

STRICT_FINAL_BLS = [
    "LS_BLS_Unemployment_Rate_Final",
    "LS_BLS_Employment_Growth_12M_Final",
    "LS_BLS_Unemployment_Rate_Change_12M_Final",
]

REGULAR_FINAL_HOUSING = [
    "Housing_HPI",
    "Housing_HPI_Growth_12M",
]

REGULAR_FINAL_BLS = [
    "BLS_Unemployment_Rate_Final",
    "BLS_Employment_Growth_12M_Final",
    "BLS_Unemployment_Rate_Change_12M_Final",
]

def complete_optional_group(cols, name):
    present = [c for c in cols if c in df.columns]
    if present and len(present) != len(cols):
        raise AssertionError(
            f"FAIL: {name} is only partially present. "
            f"Present={present}; missing={sorted(set(cols)-set(present))}"
        )
    return bool(present)

HAS_REG_HOUSING = complete_optional_group(REGULAR_FINAL_HOUSING, "Regular final housing")
HAS_REG_BLS = complete_optional_group(REGULAR_FINAL_BLS, "Regular final BLS")

required = (
    LOAN_TRAIN_MEDIAN + [DTI_COL, DTI_INDICATOR]
    + CATEGORICAL_EXPLICIT_MISSING + HIGH_MISSING_ABLATION
    + TEMPORAL_SUPPORT_EXCLUDE
    + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
)
absent = [c for c in required if c not in df.columns]
assert not absent, f"FAIL: required treatment columns absent: {absent}"

print("Regular final housing detected:", HAS_REG_HOUSING)
print("Regular final BLS detected    :", HAS_REG_BLS)
print("PASS — required treatment fields detected.")


Regular final housing detected: True
Regular final BLS detected    : True
PASS — required treatment fields detected.


## 3. Global missingness inventory

In [4]:
missing_inventory = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
    "unique_non_null": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["missing_count", "column"], ascending=[False, True])

missing_only = missing_inventory.query("missing_count > 0").copy()

print(f"Total columns: {df.shape[1]}")
print(f"Columns still containing missing values: {len(missing_only)}")
display(missing_only)

missing_inventory.to_csv(QA_DIR / "05_all_column_missingness.csv", index=False)
missing_only.to_csv(QA_DIR / "05_remaining_missingness.csv", index=False)


Total columns: 110
Columns still containing missing values: 44


,column,dtype,missing_count,missing_pct,unique_non_null
28,Property_Valuation_Method,string,487320,68.347826,3
43,FHFA_MSA_HPI,float64,148130,20.775596,14771
44,FHFA_MSA_HPI_Growth_12M,float64,148130,20.775596,23896
64,LS_FHFA_MSA_HPI,float64,148130,20.775596,14547
65,LS_FHFA_MSA_HPI_Growth_12M,float64,148130,20.775596,23831
71,LS_BLS_Employment,float64,128686,18.048527,55872
73,LS_BLS_Employment_Growth_12M,float64,128686,18.048527,60620
72,LS_BLS_Labor_Force,float64,128686,18.048527,56090
70,LS_BLS_Unemployment_Rate,float64,128686,18.048527,260
74,LS_BLS_Unemployment_Rate_Change_12M,float64,128686,18.048527,943


## 4. Verify fields that must be complete

This is the primary treatment check. Missing values in raw/intermediate lineage fields are not included here.


In [5]:
must_be_complete = (
    LOAN_TRAIN_MEDIAN
    + [DTI_COL, DTI_INDICATOR]
    + CATEGORICAL_EXPLICIT_MISSING
    + HIGH_MISSING_ABLATION
    + STRICT_FINAL_HOUSING
    + STRICT_FINAL_BLS
    + (REGULAR_FINAL_HOUSING if HAS_REG_HOUSING else [])
    + (REGULAR_FINAL_BLS if HAS_REG_BLS else [])
)

complete_check = pd.DataFrame({
    "column": must_be_complete,
    "missing_count": [int(df[c].isna().sum()) for c in must_be_complete],
    "missing_pct": [float(df[c].isna().mean()*100) for c in must_be_complete],
})
complete_check["PASS"] = complete_check["missing_count"].eq(0)

display(complete_check)
complete_check.to_csv(QA_DIR / "05_required_complete_fields.csv", index=False)

assert complete_check["PASS"].all(), (
    "FAIL: fields that should have been treated still contain missing values:\n"
    + complete_check.loc[~complete_check["PASS"]].to_string(index=False)
)

print("PASS — all fields requiring missing-value treatment are complete.")


,column,missing_count,missing_pct,PASS
0,FICO_Score,0,0.0,True
1,MI_Percentage,0,0.0,True
2,Number_of_Units,0,0.0,True
3,Original_CLTV,0,0.0,True
4,Original_LTV,0,0.0,True
5,Original_DTI,0,0.0,True
6,DTI_Missing,0,0.0,True
7,First_Time_Homebuyer,0,0.0,True
8,MSA,0,0.0,True
9,Special_Eligibility_Program,0,0.0,True


PASS — all fields requiring missing-value treatment are complete.


## 5. Split-specific verification

In [6]:
rows = []
for split_value, part in df.groupby("Split", dropna=False):
    for c in must_be_complete:
        rows.append({
            "Split": split_value,
            "column": c,
            "rows": len(part),
            "missing_count": int(part[c].isna().sum()),
            "missing_pct": float(part[c].isna().mean()*100),
        })

split_missing = pd.DataFrame(rows)
split_fail = split_missing.query("missing_count > 0")

print("Any treated-field missingness by split?")
display(split_fail if len(split_fail) else pd.DataFrame({"Result":["None"]}))

split_missing.to_csv(QA_DIR / "05_required_fields_missingness_by_split.csv", index=False)

assert split_fail.empty, (
    "FAIL: at least one treated field still has missing values within a split."
)
print("PASS — Train / Validation / OOT / other splits are all complete for treated fields.")


Any treated-field missingness by split?


,Result
0,None


PASS — Train / Validation / OOT / other splits are all complete for treated fields.


## 6. Validate DTI missing indicator

Without the original pre-treatment dataset, the saved indicator cannot reconstruct the old missing mask directly.  
However, it must be binary, non-missing, and its count can be checked against the treatment manifest.


In [7]:
dti_values = set(df[DTI_INDICATOR].dropna().unique().tolist())
assert dti_values.issubset({0, 1, False, True}), f"FAIL: invalid DTI_Missing values: {dti_values}"
assert df[DTI_INDICATOR].notna().all()
assert df[DTI_COL].notna().all()

dti_summary = (
    df.groupby("Split", dropna=False)[DTI_INDICATOR]
      .agg(["count", "sum", "mean"])
      .reset_index()
      .rename(columns={"sum":"DTI_missing_rows", "mean":"DTI_missing_rate"})
)
display(dti_summary)

print("Total DTI rows marked originally missing:", int(df[DTI_INDICATOR].sum()))
print("PASS — DTI indicator is binary and DTI is fully imputed.")


,Split,count,DTI_missing_rows,DTI_missing_rate
0,OOT Test,99719,6,0.000060
1,Other Period,404240,60425,0.149478
2,Train,160003,16001,0.100004
3,Validation,49038,1594,0.032505


Total DTI rows marked originally missing: 78026
PASS — DTI indicator is binary and DTI is fully imputed.


## 7. Verify explicit categorical Missing levels

In [8]:
cat_rows = []
for c in CATEGORICAL_EXPLICIT_MISSING + HIGH_MISSING_ABLATION:
    vals = df[c].astype("string")
    cat_rows.append({
        "column": c,
        "remaining_nulls": int(vals.isna().sum()),
        "explicit_Missing_count": int(vals.eq("Missing").sum()),
        "n_unique": int(vals.nunique(dropna=True)),
    })

cat_check = pd.DataFrame(cat_rows)
display(cat_check)
cat_check.to_csv(QA_DIR / "05_categorical_missing_representation.csv", index=False)

assert cat_check["remaining_nulls"].eq(0).all()
print("PASS — selected categorical fields contain no nulls and use explicit Missing where applicable.")


,column,remaining_nulls,explicit_Missing_count,n_unique
0,First_Time_Homebuyer,0,13,3
1,MSA,0,89555,456
2,Special_Eligibility_Program,0,683485,4


PASS — selected categorical fields contain no nulls and use explicit Missing where applicable.


## 8. Verify final macro variables

Final resolved Strict/LS macro predictors must be complete. Regular-PIT final benchmark fields are checked when they exist.


In [9]:
macro_groups = {
    "Strict final housing": STRICT_FINAL_HOUSING,
    "Strict final BLS": STRICT_FINAL_BLS,
}
if HAS_REG_HOUSING:
    macro_groups["Regular final housing"] = REGULAR_FINAL_HOUSING
if HAS_REG_BLS:
    macro_groups["Regular final BLS"] = REGULAR_FINAL_BLS

macro_rows = []
for group, cols in macro_groups.items():
    for c in cols:
        macro_rows.append({
            "group": group,
            "column": c,
            "missing_count": int(df[c].isna().sum()),
            "missing_pct": float(df[c].isna().mean()*100),
        })

macro_check = pd.DataFrame(macro_rows)
macro_check["PASS"] = macro_check["missing_count"].eq(0)
display(macro_check)
macro_check.to_csv(QA_DIR / "05_final_macro_completeness.csv", index=False)

assert macro_check["PASS"].all()
print("PASS — all detected final resolved macro predictors are complete.")


,group,column,missing_count,missing_pct,PASS
0,Strict final housing,LS_Housing_HPI,0,0.0,True
1,Strict final housing,LS_Housing_HPI_Growth_12M,0,0.0,True
2,Strict final BLS,LS_BLS_Unemployment_Rate_Final,0,0.0,True
3,Strict final BLS,LS_BLS_Employment_Growth_12M_Final,0,0.0,True
4,Strict final BLS,LS_BLS_Unemployment_Rate_Change_12M_Final,0,0.0,True
5,Regular final housing,Housing_HPI,0,0.0,True
6,Regular final housing,Housing_HPI_Growth_12M,0,0.0,True
7,Regular final BLS,BLS_Unemployment_Rate_Final,0,0.0,True
8,Regular final BLS,BLS_Employment_Growth_12M_Final,0,0.0,True
9,Regular final BLS,BLS_Unemployment_Rate_Change_12M_Final,0,0.0,True


PASS — all detected final resolved macro predictors are complete.


## 9. Detect and audit residual-missing indicators

In [10]:
indicator_cols = [
    c for c in df.columns
    if c == "DTI_Missing"
    or "ResidualMissing" in c
    or c.endswith("_Missing")
]

indicator_rows = []
for c in indicator_cols:
    s = df[c]
    vals = set(pd.Series(s.dropna().unique()).tolist())
    binary = vals.issubset({0, 1, False, True})
    indicator_rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "missing_count": int(s.isna().sum()),
        "sum": int(s.fillna(0).astype(int).sum()) if binary else np.nan,
        "binary_0_1": binary,
    })

indicator_inventory = pd.DataFrame(indicator_rows)
display(indicator_inventory)
indicator_inventory.to_csv(QA_DIR / "05_missing_indicator_inventory.csv", index=False)

assert indicator_inventory["missing_count"].eq(0).all()
assert indicator_inventory["binary_0_1"].all()
print("PASS — all detected missing indicators are non-null and binary.")


,column,dtype,missing_count,sum,binary_0_1
0,DTI_Missing,int8,0,78026,True
1,LS_Housing_ResidualMissing,int8,0,329,True
2,LS_BLS_ResidualMissing,int8,0,333,True
3,Housing_Regular_ResidualMissing,int8,0,329,True
4,BLS_Regular_ResidualMissing,int8,0,412,True


PASS — all detected missing indicators are non-null and binary.


## 10. Confirm intentional missingness is not mistaken for failure

`Property_Valuation_Method` is explicitly excluded from the primary model and was intentionally not imputed. Raw/intermediate macro, PIT-date, join, fallback-lineage, and provenance columns may also retain missing values.


In [11]:
intentional_rows = []

for _, r in missing_only.iterrows():
    c = r["column"]
    if c in TEMPORAL_SUPPORT_EXCLUDE:
        role = "TEMPORAL_SUPPORT_EXCLUDE"
    elif c.startswith("_State_") or c.startswith("Strict_PIT_") or c.startswith("PIT_"):
        role = "AUDIT_OR_LINEAGE"
    elif c.startswith("FHFA_") or c.startswith("LS_FHFA_"):
        role = "RAW_OR_INTERMEDIATE_HPI"
    elif c.startswith("BLS_") or c.startswith("LS_BLS_"):
        role = "RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE"
    elif "Fallback" in c or "Geography_Source" in c or c.startswith("Freddie_") or c.startswith("LS_Freddie_"):
        role = "FALLBACK_OR_GEOGRAPHY_LINEAGE"
    else:
        role = "REVIEW_MANUALLY"

    intentional_rows.append({
        "column": c,
        "missing_count": int(r["missing_count"]),
        "missing_pct": float(r["missing_pct"]),
        "audit_classification": role,
    })

remaining_classification = pd.DataFrame(intentional_rows)
display(remaining_classification)
remaining_classification.to_csv(
    QA_DIR / "05_remaining_missingness_classification.csv", index=False
)

manual_review = remaining_classification.query("audit_classification == 'REVIEW_MANUALLY'")
if len(manual_review):
    print("WARNING — remaining missing columns not automatically recognized as intentional:")
    display(manual_review)
else:
    print("PASS — all remaining missing columns fall into recognized excluded/lineage groups.")


,column,missing_count,missing_pct,audit_classification
0,Property_Valuation_Method,487320,68.347826,TEMPORAL_SUPPORT_EXCLUDE
1,FHFA_MSA_HPI,148130,20.775596,RAW_OR_INTERMEDIATE_HPI
2,FHFA_MSA_HPI_Growth_12M,148130,20.775596,RAW_OR_INTERMEDIATE_HPI
3,LS_FHFA_MSA_HPI,148130,20.775596,RAW_OR_INTERMEDIATE_HPI
4,LS_FHFA_MSA_HPI_Growth_12M,148130,20.775596,RAW_OR_INTERMEDIATE_HPI
5,LS_BLS_Employment,128686,18.048527,RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE
6,LS_BLS_Employment_Growth_12M,128686,18.048527,RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE
7,LS_BLS_Labor_Force,128686,18.048527,RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE
8,LS_BLS_Unemployment_Rate,128686,18.048527,RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE
9,LS_BLS_Unemployment_Rate_Change_12M,128686,18.048527,RAW_OR_INTERMEDIATE_BLS_OR_PROVENANCE


PASS — all remaining missing columns fall into recognized excluded/lineage groups.


## 11. Compare with Notebook 04 treatment manifest

In [12]:
manifest_checks = []

if TREATMENT_MANIFEST.exists():
    manifest = json.loads(TREATMENT_MANIFEST.read_text(encoding="utf-8"))

    expected_shape = manifest.get("output_shape")
    if expected_shape:
        ok = list(df.shape) == expected_shape
        manifest_checks.append({
            "check": "output_shape",
            "expected": str(expected_shape),
            "observed": str(list(df.shape)),
            "PASS": ok,
        })

    expected_indicators = manifest.get("created_indicators", [])
    missing_indicators = [c for c in expected_indicators if c not in df.columns]
    manifest_checks.append({
        "check": "created_indicators_present",
        "expected": str(expected_indicators),
        "observed": f"missing={missing_indicators}",
        "PASS": len(missing_indicators) == 0,
    })

    expected_train = manifest.get("train_rows")
    if expected_train is not None:
        manifest_checks.append({
            "check": "train_rows",
            "expected": str(expected_train),
            "observed": str(int(train_mask.sum())),
            "PASS": int(train_mask.sum()) == int(expected_train),
        })

    manifest_table = pd.DataFrame(manifest_checks)
    display(manifest_table)
    manifest_table.to_csv(QA_DIR / "05_manifest_crosscheck.csv", index=False)

    assert manifest_table["PASS"].all(), "FAIL: treatment manifest cross-check failed."
    print("PASS — saved dataset is consistent with Notebook 04 treatment manifest.")
else:
    print("NOTE — Notebook 04 treatment manifest was not found.")
    print("Dataset-level verification continues, but exact treatment-run metadata cannot be cross-checked.")


,check,expected,observed,PASS
0,output_shape,"[713000, 110]","[713000, 110]",True
1,created_indicators_present,"['DTI_Missing', 'LS_Housing_ResidualMissing', ...",missing=[],True
2,train_rows,160003,160003,True


PASS — saved dataset is consistent with Notebook 04 treatment manifest.


## 12. Train-median consistency check

When the Notebook 04 manifest exists, this section checks a strong invariant:

> Every value that had to be filled by the saved treatment run should use the frozen Train median recorded in the manifest.

Because the pre-treatment missing mask is no longer present for ordinary numeric fields, this cannot prove which individual rows were originally missing. It does verify that the recorded Train medians are valid and that all treated columns contain those frozen values where applicable.


In [13]:
median_validation_rows = []

if TREATMENT_MANIFEST.exists():
    manifest = json.loads(TREATMENT_MANIFEST.read_text(encoding="utf-8"))
    medians = manifest.get("train_medians", {})

    for c, med in medians.items():
        if c not in df.columns:
            median_validation_rows.append({
                "column": c, "manifest_train_median": med,
                "column_present": False, "count_equal_to_median": np.nan
            })
            continue

        numeric = pd.to_numeric(df[c], errors="coerce")
        count_equal = int(np.isclose(numeric.to_numpy(dtype=float), float(med), equal_nan=False).sum())
        median_validation_rows.append({
            "column": c,
            "manifest_train_median": float(med),
            "column_present": True,
            "count_equal_to_median": count_equal,
        })

    median_validation = pd.DataFrame(median_validation_rows)
    display(median_validation)
    median_validation.to_csv(QA_DIR / "05_manifest_median_presence_check.csv", index=False)
else:
    print("Manifest unavailable — skipped.")


,column,manifest_train_median,column_present,count_equal_to_median
0,FICO_Score,757.000000,True,5123
1,MI_Percentage,0.000000,True,544000
2,Number_of_Units,1.000000,True,698473
3,Original_CLTV,80.000000,True,130819
4,Original_LTV,79.000000,True,11650
5,Original_DTI,35.000000,True,99083
6,LS_Housing_HPI,193.720000,True,374
7,LS_Housing_HPI_Growth_12M,0.052081,True,415
8,LS_BLS_Unemployment_Rate_Final,5.200000,True,10600
9,LS_BLS_Employment_Growth_12M_Final,0.019572,True,368


## 13. Final verdict

In [14]:
checks = []

def add_check(name, passed, detail):
    checks.append({"check": name, "PASS": bool(passed), "detail": detail})

add_check("Core identifiers / split / target valid", True, "Hard assertions passed")
add_check("All required treated fields complete", bool(complete_check["PASS"].all()),
          f"{int((~complete_check['PASS']).sum())} failures")
add_check("No treated-field missingness in any split", split_fail.empty,
          f"{len(split_fail)} split-field failures")
add_check("DTI indicator binary and DTI complete",
          df[DTI_INDICATOR].notna().all() and df[DTI_COL].notna().all(),
          f"DTI_Missing sum={int(df[DTI_INDICATOR].sum())}")
add_check("Categorical explicit-missing treatment complete",
          cat_check["remaining_nulls"].eq(0).all(),
          "Selected categorical fields have zero nulls")
add_check("Final macro fields complete", macro_check["PASS"].all(),
          f"{int((~macro_check['PASS']).sum())} failures")
add_check("Missing indicators valid",
          indicator_inventory["missing_count"].eq(0).all() and indicator_inventory["binary_0_1"].all(),
          f"{len(indicator_inventory)} indicators checked")
add_check("Remaining missingness recognized",
          manual_review.empty,
          f"{len(manual_review)} columns require manual review")

if TREATMENT_MANIFEST.exists() and manifest_checks:
    add_check("Notebook 04 manifest consistency",
              pd.DataFrame(manifest_checks)["PASS"].all(),
              "Shape / indicator / Train-row checks")

final_report = pd.DataFrame(checks)
display(final_report)
final_report.to_csv(QA_DIR / "05_final_verification_report.csv", index=False)

overall_pass = final_report["PASS"].all()

print("=" * 100)
if overall_pass:
    print("FINAL RESULT: PASS")
    print("The treated dataset satisfies the missing-value treatment checks implemented in this audit.")
else:
    print("FINAL RESULT: FAIL / REVIEW REQUIRED")
    print("At least one frozen-policy verification check failed. Inspect the table above.")
print("=" * 100)

summary = {
    "dataset": FILE_NAME,
    "shape": [int(df.shape[0]), int(df.shape[1])],
    "overall_pass": bool(overall_pass),
    "remaining_missing_columns": int(len(missing_only)),
    "manual_review_remaining_columns": manual_review["column"].tolist() if len(manual_review) else [],
    "checks": final_report.to_dict(orient="records"),
}
(QA_DIR / "05_final_verification_report.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("QA files saved to:", QA_DIR.resolve())


,check,PASS,detail
0,Core identifiers / split / target valid,True,Hard assertions passed
1,All required treated fields complete,True,0 failures
2,No treated-field missingness in any split,True,0 split-field failures
3,DTI indicator binary and DTI complete,True,DTI_Missing sum=78026
4,Categorical explicit-missing treatment complete,True,Selected categorical fields have zero nulls
5,Final macro fields complete,True,0 failures
6,Missing indicators valid,True,5 indicators checked
7,Remaining missingness recognized,True,0 columns require manual review
8,Notebook 04 manifest consistency,True,Shape / indicator / Train-row checks


FINAL RESULT: PASS
The treated dataset satisfies the missing-value treatment checks implemented in this audit.
QA files saved to: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\05_missing_value_treatment_verification
